# Chip Picker

Give it a color (`'red'`, `'green'`, `'black'`) and the arm picks that chip out of the blue tape **my chips** box, then can drop it on the bet square.

How it works:
1. Arm goes to the `mychips` pose and takes a picture.
2. Finds the **white square** inside the blue tape. Its 4 corners are at known table spots, so they give a **homography** (pixel -> table x, y). Corners are labeled automatically.
3. Finds chip stacks of the chosen color. Each one must be a **4 cm circle**; its center becomes the target.
4. **IK** (built on `fk_Dofbot` from Part 1) turns x, y, z into joint angles.
5. Hover -> down -> close gripper -> lift -> (bet square) -> down -> open -> lift -> home.

Units: meters, same base frame as `fk_Dofbot` (+y = forward at base 90).

**Run order the first time:** setup -> settings -> helpers -> color test -> check the view -> calibrate -> find chips (no motion) -> IK check -> hover test -> gripper test -> pick.

In [ ]:
%run ../Part1/DofbotKinematics.ipynb

import json
import os
import time
import cv2
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()

CAMERA_INDEX = 0   # try 1 if you get no frames
cap = cv2.VideoCapture(CAMERA_INDEX)
time.sleep(1)      # let the camera warm up

with open('poses.json') as f:
    poses = json.load(f)

## Settings (edit these)

In [ ]:
# --- Table layout, in the FK base frame (meters). +y = forward at base 90 ---
# The WHITE square inside the blue tape (inner edge of the tape):
# corner closest to the robot and the diagonal corner farthest away. Sides parallel to the robot's x / y axes.
BOX_MIN = (0.05, 0.15)
BOX_MAX = (0.18, 0.225)
# Corner names: A = near corner, B = (far x, near y), C = far corner, D = (near x, far y)

CHIP_DIAMETER = 0.04     # every stack is a 4 cm circle
CHIP_HEIGHT = 0.017      # 1.7 cm, a taped stack we move as one piece

# Where to drop chips on the bet square (x, y) in meters. MEASURE THIS with a ruler.
BET_XY = None            # e.g. (-0.10, 0.18)

# --- Motion heights for the gripper tip (meters above the table) ---
HOVER_Z = 0.06           # travel height above chips
GRAB_Z = 0.006           # fingertips this low when closing on a chip. Raise if it hits the table
PLACE_Z = 0.025          # height to let go over the bet square

# --- Gripper (servo 6). Find these with camera_pose_tester / the gripper test cell below ---
GRIP_OPEN = 40           # wide enough for a 4 cm chip
GRIP_CLOSED = 135        # holding a chip firmly

# --- Colors in HSV (OpenCV: H 0-180, S 0-255, V 0-255). Tune with the color test cell ---
COLORS = {
    'blue':  [((95, 80, 40), (130, 255, 255))],     # tape
    'white': [((0, 0, 160), (180, 60, 255))],       # the box floor
    'red':   [((0, 100, 70), (10, 255, 255)), ((170, 100, 70), (180, 255, 255))],  # red wraps around H=0
    'green': [((40, 70, 50), (85, 255, 255))],
    'black': [((0, 0, 0), (180, 255, 60))],
}

CALIB_FILE = 'chip_box_calib.json'
calib = None             # filled in by step 2b

## Vision helpers

In [ ]:
def take_picture():
    global cap
    if not cap.isOpened():
        cap = cv2.VideoCapture(CAMERA_INDEX)
        time.sleep(1)
    # Throw away a few old frames so we get the current view
    for _ in range(5):
        cap.read()
    ok, frame = cap.read()
    return frame if ok else None


def color_mask(frame, color):
    hsv = cv2.cvtColor(cv2.GaussianBlur(frame, (5, 5), 0), cv2.COLOR_BGR2HSV)
    mask = np.zeros(hsv.shape[:2], np.uint8)
    for lo, hi in COLORS[color]:
        mask |= cv2.inRange(hsv, np.array(lo), np.array(hi))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    return mask


def box_world_corners():
    """White square corners A, B, C, D in meters (A = near, C = far). Counterclockwise seen from above."""
    (x0, y0), (x1, y1) = BOX_MIN, BOX_MAX
    return np.array([(x0, y0), (x1, y0), (x1, y1), (x0, y1)], np.float32)


def line_through(pts):
    """Robust straight line through points (Huber fit ignores bumps). Returns (point, direction)."""
    vx, vy, x0, y0 = cv2.fitLine(pts.astype(np.float32), cv2.DIST_HUBER, 0, 0.01, 0.01).flatten()
    return np.array([x0, y0]), np.array([vx, vy])


def intersect(l1, l2):
    (p1, d1), (p2, d2) = l1, l2
    A = np.array([d1, -d2]).T
    if abs(np.linalg.det(A)) < 1e-9:
        return None
    t = np.linalg.solve(A, p2 - p1)[0]
    return p1 + t * d1


def quad_from_contour(contour):
    """4 corners of a 4-sided outline: rough corners first, then a straight line fitted to each side
    (so chips poking over the tape don't bend the corners), then corners = where the sides cross."""
    hull = cv2.convexHull(contour)
    rough = None
    for eps in (0.01, 0.02, 0.03, 0.05, 0.08):
        approx = cv2.approxPolyDP(hull, eps * cv2.arcLength(hull, True), True)
        if len(approx) == 4:
            rough = approx.reshape(4, 2).astype(np.float32)
            break
    if rough is None:
        return cv2.boxPoints(cv2.minAreaRect(contour)).astype(np.float32)
    pts = contour.reshape(-1, 2).astype(np.float32)
    lines = []
    for i in range(4):
        a, b = rough[i], rough[(i + 1) % 4]
        d = (b - a) / (np.linalg.norm(b - a) + 1e-9)
        rel = pts - a
        along = rel @ d
        off = np.abs(rel @ np.array([-d[1], d[0]]))
        L = np.linalg.norm(b - a)
        near = pts[(off < 6) & (along > 0.1 * L) & (along < 0.9 * L)]   # middle of the side only
        lines.append(line_through(near) if len(near) >= 5 else (a, d))
    corners = [intersect(lines[i - 1], lines[i]) for i in range(4)]
    if any(c is None for c in corners):
        return rough
    return np.array(corners, np.float32)


def find_white_box(frame, min_area=3000):
    """Finds the white square inside the blue tape. Returns its 4 corners (pixels), or None.
    Inside = everything within the tape's outer outline that isn't tape (white floor + any chips).
    Works even if a tall stack hides part of the tape."""
    blue = color_mask(frame, 'blue')
    contours = cv2.findContours(blue, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    if not contours:
        return None
    tape = cv2.convexHull(np.vstack([c for c in contours if cv2.contourArea(c) > 200] or [max(contours, key=cv2.contourArea)]))
    inside = np.zeros(blue.shape, np.uint8)
    cv2.fillPoly(inside, [tape], 255)
    inside &= ~blue
    inside = cv2.morphologyEx(inside, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours = cv2.findContours(inside, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[-2]
    contours = [c for c in contours if cv2.contourArea(c) >= min_area]
    if not contours:
        return None
    return quad_from_contour(max(contours, key=cv2.contourArea))


def label_corners(corners_px, cam_xy, frame_shape):
    """Puts the 4 image corners in A, B, C, D order automatically.
    Seen by a camera looking down, the box's counterclockwise order stays counterclockwise in the picture, so only
    4 orders are possible. The right one is where the bottom-middle of the picture lands closest to the camera
    (from FK): the bottom of the picture is always the table nearest the camera."""
    c = np.array(corners_px, np.float32)
    center = c.mean(0)
    c = c[np.argsort(-np.arctan2(c[:, 1] - center[1], c[:, 0] - center[0]))]  # counterclockwise on screen
    h, w = frame_shape[:2]
    bottom = np.array([[[w / 2, h - 1]]], np.float32)
    def score(k):
        H = cv2.getPerspectiveTransform(np.roll(c, -k, axis=0), box_world_corners())
        p = cv2.perspectiveTransform(bottom, H)[0, 0]
        return np.hypot(*(p - np.asarray(cam_xy)))
    best = min(range(4), key=score)
    return np.roll(c, -best, axis=0)


def box_in_view(corners_px, shape, margin=3):
    h, w = shape[:2]
    return bool(np.all((corners_px[:, 0] > margin) & (corners_px[:, 0] < w - margin) &
                       (corners_px[:, 1] > margin) & (corners_px[:, 1] < h - margin)))


def calibrate_from_frame(frame, cam_xyz):
    """White square -> homography (pixel -> table meters). Returns (H, labeled corners) or (None, corners)."""
    corners = find_white_box(frame)
    if corners is None:
        return None, None
    corners = label_corners(corners, cam_xyz[:2], frame.shape)
    if not box_in_view(corners, frame.shape):
        return None, corners
    return cv2.getPerspectiveTransform(corners, box_world_corners()), corners


def px_to_table(H, pts):
    pts = np.array(pts, np.float32).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, H).reshape(-1, 2)


def table_to_px(H, pts):
    return px_to_table(np.linalg.inv(H), pts)


def top_to_table_factor(cam_xyz):
    """The camera sees chip TOPS (CHIP_HEIGHT up). A top point maps onto the table too far from the camera;
    scaling toward the camera by this factor fixes it."""
    return (cam_xyz[2] - CHIP_HEIGHT) / cam_xyz[2]


def fit_circle(pts, radius, cam_xy, res=0.0005, tol=0.003):
    """Finds the center of a 4 cm circle from a stack's outline (table meters, chip-top plane).
    From the camera's angle the FAR half of the top rim is always visible, while the near side is
    smeared by the stack's side wall. So: try every center near the blob and keep the one whose far
    half-circle lands on the outline best.
    Returns (center, coverage) where coverage = fraction of that far half-circle on the outline (1.0 = perfect)."""
    lo = pts.min(0) - radius
    hi = pts.max(0) + radius
    w, h = (np.ceil((hi - lo) / res) + 1).astype(int)
    # Distance (meters) from every grid cell to the nearest outline point
    canvas = np.full((h, w), 255, np.uint8)
    cv2.polylines(canvas, [((pts - lo) / res).round().astype(np.int32)], True, 0, 1)
    dist = cv2.distanceTransform(canvas, cv2.DIST_L2, 3) * res

    away = np.arctan2(*(pts.mean(0) - cam_xy)[::-1])
    t = away + np.linspace(-np.pi / 2, np.pi / 2, 25)
    ring = radius * np.c_[np.cos(t), np.sin(t)]

    def coverage(centers):
        rim = ((centers[:, None, :] + ring[None]) - lo) / res
        u = np.clip(rim[..., 0].round().astype(int), 0, w - 1)
        v = np.clip(rim[..., 1].round().astype(int), 0, h - 1)
        d = dist[v, u]
        return (d < tol).mean(1) - d.mean(1)          # most rim on the outline, then closest

    best = pts.mean(0)
    for span, step in ((0.02, 0.001), (0.0015, 0.00025)):   # coarse grid, then fine
        g = np.arange(-span, span + 1e-9, step)
        cand = best + np.stack(np.meshgrid(g, g), -1).reshape(-1, 2)
        best = cand[coverage(cand).argmax()]
    rim = ((best + ring) - lo) / res
    d = dist[np.clip(rim[:, 1].round().astype(int), 0, h - 1), np.clip(rim[:, 0].round().astype(int), 0, w - 1)]
    return best, float((d < tol).mean())


def box_area_px(H, cam_xyz, shape, grow=0.012):
    """Image area where chip tops in the box can show up (box grown a bit: tall stacks lean outward in the picture)."""
    k = top_to_table_factor(cam_xyz)
    cam = np.array(cam_xyz[:2], float)
    (x0, y0), (x1, y1) = BOX_MIN, BOX_MAX
    top = np.array([(x0 - grow, y0 - grow), (x1 + grow, y0 - grow), (x1 + grow, y1 + grow), (x0 - grow, y1 + grow)])
    poly = table_to_px(H, cam + (top - cam) / k)
    area = np.zeros(shape[:2], np.uint8)
    cv2.fillPoly(area, [poly.astype(np.int32)], 255)
    return area


def find_chips(frame, color, H, cam_xyz, min_rim=0.8):
    """Chip stacks of one color inside the box. Each one has to be a 4 cm circle.
    Returns a list of dicts, closest to the robot first."""
    k = top_to_table_factor(cam_xyz)
    cam = np.array(cam_xyz[:2], float)
    R = CHIP_DIAMETER / 2
    mask = color_mask(frame, color) & box_area_px(H, cam_xyz, frame.shape)
    chips = []
    for contour in cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[-2]:
        if cv2.contourArea(contour) < 150:
            continue
        # outline -> table meters, on the plane of the chip tops
        pts = cam + (px_to_table(H, contour[:, 0, :]) - cam) * k
        size = 2 * np.sqrt(cv2.contourArea(pts.astype(np.float32)) / np.pi)
        if not (0.75 * CHIP_DIAMETER <= size <= 1.6 * CHIP_DIAMETER):
            continue                                           # way too small / big for one stack
        center, rim = fit_circle(pts, R, cam)
        if rim < min_rim:
            continue                                           # outline doesn't trace a 4 cm circle
        x, y = center
        if not (BOX_MIN[0] <= x <= BOX_MAX[0] and BOX_MIN[1] <= y <= BOX_MAX[1]):
            continue                                           # center outside the white square
        chips.append({'xy': (float(x), float(y)), 'size_cm': size * 100, 'rim': rim})
    chips.sort(key=lambda ch: np.hypot(*ch['xy']))             # closest to the base = easiest to reach
    return chips


def chip_circle_px(H, cam_xyz, xy):
    """The 4 cm circle (top of the stack) drawn back onto the picture."""
    k = top_to_table_factor(cam_xyz)
    cam = np.array(cam_xyz[:2], float)
    t = np.linspace(0, 2 * np.pi, 48)
    top = np.c_[xy[0] + CHIP_DIAMETER / 2 * np.cos(t), xy[1] + CHIP_DIAMETER / 2 * np.sin(t)]
    return table_to_px(H, cam + (top - cam) / k)


def show(frame, chips=(), title='', H=None, corners=None, cam_xyz=None):
    img = frame.copy()
    if corners is not None:
        cv2.polylines(img, [corners.astype(np.int32)], True, (255, 0, 255), 2)
        for (u, v), name in zip(corners, 'ABCD'):
            cv2.putText(img, name, (int(u) + 5, int(v) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)
    for i, ch in enumerate(chips):
        ring = chip_circle_px(H, cam_xyz, ch['xy'])
        cv2.polylines(img, [ring.astype(np.int32)], True, (0, 255, 0), 2)
        u, v = ring.mean(0).astype(int)
        cv2.circle(img, (u, v), 4, (0, 0, 255), -1)
        cv2.putText(img, f"#{i} ({ch['xy'][0]*100:.1f},{ch['xy'][1]*100:.1f})cm", (u + 6, v),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 160, 0), 2)
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

## IK helpers

In [ ]:
# Same chain as fk_Dofbot (Part 1), written with numpy arrays so IK can run many guesses at once, fast.
_P01 = (l0 + l1) * ez
_P23 = l2 * ex
_P34 = -l3 * ez
_P5T = -(l4 + l5) * ex


def _rz(t):
    c, s, o, z = np.cos(t), np.sin(t), np.ones_like(t), np.zeros_like(t)
    return np.stack([np.stack([c, -s, z], -1), np.stack([s, c, z], -1), np.stack([z, z, o], -1)], -2)


def _ry(t):
    c, s, o, z = np.cos(t), np.sin(t), np.ones_like(t), np.zeros_like(t)
    return np.stack([np.stack([c, z, s], -1), np.stack([z, o, z], -1), np.stack([-s, z, c], -1)], -2)


def fk_fast(Q):
    """Gripper tip position and pointing direction for joints 1-4 (degrees), for many joint sets at once.
    Same math as fk_Dofbot. Wrist roll (joint 5) only spins the gripper about its own axis,
    so it doesn't change the tip or the pointing direction."""
    t = np.radians(np.atleast_2d(Q))
    R01 = _rz(t[:, 0])
    R02 = R01 @ _ry(-t[:, 1])
    R03 = R02 @ _ry(-t[:, 2])
    R04 = R03 @ _ry(-t[:, 3])
    pos = _P01 + R02 @ _P23 + R03 @ _P34 + R04 @ _P5T
    return pos, -R04[:, :, 0]                       # the gripper points along -x of the tool frame


DOWN = np.array([0, 0, -1.0])
Q_LO, Q_HI = np.zeros(4), np.full(4, 180.0)


def ik_residual(Q, target, w_down):
    pos, pointing = fk_fast(Q)
    return np.concatenate([pos - target, w_down * (pointing - DOWN)], 1)


def ik_Dofbot(target, w_down=0.003, iters=80):
    """Joint angles [q1..q5] (deg) that put the gripper tip at target (x, y, z in meters). q5 stays 90.
    Position comes first; among good answers it prefers pointing the gripper straight down.
    Same idea as the Jacobian in sensitivity.py (nudge each joint, see how the tip moves, step toward the
    target), run from 12 starting guesses at the same time."""
    target = np.asarray(target, float)
    base = np.degrees(np.arctan2(target[1], target[0]))
    Q = np.array([[base, a, b, c] for a in (10, 40, 80) for b in (60, 100) for c in (0, 40)], float)
    Q = np.clip(Q, Q_LO, Q_HI)
    eye = 1e-6 * np.eye(4)
    for _ in range(iters):
        r = ik_residual(Q, target, w_down)                                    # (12, 6)
        J = np.stack([(ik_residual(Q + 0.01 * np.eye(4)[i], target, w_down) - r) / 0.01
                      for i in range(4)], 2)                                  # (12, 6, 4)
        Jt = J.transpose(0, 2, 1)
        step = -np.linalg.solve(Jt @ J + eye, (Jt @ r[..., None]))[..., 0]
        Q = np.clip(Q + np.clip(step, -10, 10), Q_LO, Q_HI)
        if np.abs(step).max() < 1e-3:
            break
    pos, pointing = fk_fast(Q)
    err = np.linalg.norm(pos - target, axis=1)
    pitch = np.degrees(np.arccos(np.clip(pointing[:, 2], -1, 1)))           # like pitch_from_vertical: 180 = down
    best = np.argmin(err + 0.00005 * (180 - pitch))                         # 1 mm of error ~ 20 deg less tilt
    q_out = [float(round(a, 1)) for a in Q[best]] + [90.0]
    # Double-check the answer with the real Part 1 FK
    Rot, Pot = fk_Dofbot(q_out)
    return q_out, float(np.linalg.norm(Pot - target)), pitch_from_vertical(Rot)


# Sanity check: fk_fast must agree with fk_Dofbot
for _q in ([90, 90, 90, 90], [60, 117, 0, 0], [45, 30, 120, 60]):
    _R, _P = fk_Dofbot(list(_q) + [90])
    _p, _d = fk_fast(_q)
    assert np.allclose(_p[0], _P, atol=1e-9) and np.allclose(_d[0], -_R.as_matrix()[:, 0], atol=1e-9), "fk_fast != fk_Dofbot"
print("fk_fast matches fk_Dofbot")

## Motion helpers

In [ ]:
last_q = [90, 90, 90, 90, 90]   # last joint angles we sent (used to open/close without moving)


def move(q, grip, ms=1500):
    # q = [q1..q5] in degrees, grip = servo 6
    global last_q
    last_q = list(q[:5])
    arm.Arm_serial_servo_write6(*q[:5], grip, ms)
    time.sleep(ms / 1000 + 0.5)


def move_to_xyz(xyz, grip, ms=1500, max_err=0.005):
    q, err, pitch = ik_Dofbot(xyz)
    print(f"  -> ({xyz[0]*100:.1f}, {xyz[1]*100:.1f}, {xyz[2]*100:.1f}) cm  q={q}  err={err*1000:.1f} mm  pitch={pitch:.0f}")
    if err > max_err:
        raise RuntimeError(f"Can't reach that spot (off by {err*1000:.0f} mm). Pick a chip closer to the robot.")
    move(q, grip, ms)
    return q


def look_for_chips(color, show_it=True):
    """Go to the mychips pose and find chips of one color in the box.
    Uses the saved calibration (taken with the box empty, so no chips hide the tape).
    If there isn't one yet, finds the white square in this picture instead."""
    move(poses['mychips'], GRIP_OPEN)
    frame = take_picture()
    if frame is None:
        raise RuntimeError("No camera frame")
    _, cam = fk_Dofbot(poses['mychips'])          # camera ~ at the gripper
    if calib is not None:
        H, corners = np.array(calib['H']), np.array(calib['corners'], np.float32)
    else:
        H, corners = calibrate_from_frame(frame, cam)
        if H is None:
            show(frame, title='White square not found', corners=corners)
            raise RuntimeError("Can't see the whole white square. Do steps 2a / 2b with the box empty.")
    chips = find_chips(frame, color, H, cam)
    if show_it:
        show(frame, chips, f"{color}: {len(chips)} chip(s)", H, corners, cam)
    return chips


def pick_chip(color):
    chips = look_for_chips(color)
    if not chips:
        print(f"No {color} chips in the box")
        return False
    x, y = chips[0]['xy']
    print(f"Picking {color} chip at ({x*100:.1f}, {y*100:.1f}) cm")
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN)            # above the chip
    move_to_xyz((x, y, GRAB_Z), GRIP_OPEN, ms=1000)    # down around it
    move(last_q, GRIP_CLOSED, ms=800)             # close
    move_to_xyz((x, y, HOVER_Z), GRIP_CLOSED, ms=1000) # lift
    return True


def place_on_bet():
    if BET_XY is None:
        raise RuntimeError("Set BET_XY (bet square center in meters) in the settings cell first.")
    x, y = BET_XY
    move_to_xyz((x, y, HOVER_Z), GRIP_CLOSED)
    move_to_xyz((x, y, PLACE_Z), GRIP_CLOSED, ms=1000)
    move(last_q, GRIP_OPEN, ms=800)               # let go
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN, ms=1000)
    move(poses['home'], GRIP_OPEN)

## 1. Color test (no arm motion besides going to `mychips`)
Shows what each color mask picks up. White = detected. If a color misses chips or grabs the table, tune its range in `COLORS`.
Black is the hardest (shadows look black too): keep its V max low.

In [ ]:
move(poses['mychips'], GRIP_OPEN)
frame = take_picture()
plt.figure(figsize=(18, 8))
plt.subplot(2, 4, 1); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.title('camera'); plt.axis('off')
for i, color in enumerate(COLORS):
    plt.subplot(2, 4, i + 2); plt.imshow(color_mask(frame, color), cmap='gray'); plt.title(color); plt.axis('off')
plt.show()

# Hover over a spot to read its HSV: print the HSV at the image center
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
h, w = hsv.shape[:2]
print("HSV at image center:", hsv[h // 2, w // 2])

## 2a. Check the view (box empty)
The camera has to see all 4 corners of the white square from the `mychips` pose.
If it says NOT fully in view, adjust `mychips` (joints 1-4) in `camera_pose_tester` / `poses.json` and rerun.

In [ ]:
move(poses['mychips'], GRIP_OPEN)
frame = take_picture()
H, corners = calibrate_from_frame(frame, fk_Dofbot(poses['mychips'])[1])
show(frame, title="All 4 corners in view" if H is not None else "NOT fully in view", corners=corners)

## 2b. Calibrate from the white square (once; redo if the pose changes)
**Empty the box first** (chips can hide the tape). Finds the white square, labels its corners automatically, and saves the pixel -> table map.
Check the purple labels: **A should be the corner closest to the robot, C the farthest**, B along +x from A, D along +y from A.
Picking uses this saved map. Redo it if the pose, camera, or box moves.

In [ ]:
move(poses['mychips'], GRIP_OPEN)
frame = take_picture()
H, corners = calibrate_from_frame(frame, fk_Dofbot(poses['mychips'])[1])
show(frame, title='White square corners', corners=corners)
if H is None:
    raise RuntimeError("Didn't find all 4 corners of the white square. Do step 2a, or check the 'blue' / 'white' masks.")

calib = {'H': H.tolist(), 'corners': corners.tolist(), 'pose': poses['mychips']}
with open(CALIB_FILE, 'w') as f:
    json.dump(calib, f)
print("Corners (pixels) A, B, C, D:", corners.round(0).tolist())
print("Saved", CALIB_FILE)

Next time you can just load it:

In [ ]:
calib = None
if os.path.exists(CALIB_FILE):
    with open(CALIB_FILE) as f:
        calib = json.load(f)
    if calib['pose'] != poses['mychips']:
        print("WARNING: the mychips pose changed since calibrating. Redo step 2b.")
print("Loaded" if calib else "No saved calibration yet", CALIB_FILE)

## 3. Find chips (no grabbing)
Check the green 4 cm circles and the cm coordinates. Put a ruler down to check one chip: it should be within ~1 cm.

In [ ]:
for color in ['red', 'green', 'black']:
    chips = look_for_chips(color)
    for i, ch in enumerate(chips):
        print(f"  {color} #{i}: x={ch['xy'][0]*100:.1f} cm  y={ch['xy'][1]*100:.1f} cm  (blob {ch['size_cm']:.1f} cm, {ch['rim']*100:.0f}% of the 4 cm rim matched)")

## 4. IK check (no motion)
Solves IK for points in the box and checks them with FK. `err` should be ~0 mm. The far corner of the box is out of reach, but chip centers never get that far (they sit 2 cm inside).

In [ ]:
for x, y in [(0.07, 0.17), (0.115, 0.1875), (0.16, 0.205), (0.07, 0.205), (0.16, 0.17)]:
    for z in (HOVER_Z, GRAB_Z):
        q, err, pitch = ik_Dofbot((x, y, z))
        print(f"({x*100:.1f}, {y*100:.1f}, {z*100:.1f}) cm -> q={q}  err={err*1000:.1f} mm  pitch={pitch:.0f}")

## 5. Hover test (safe: stays 6 cm up)
Moves above the first chip of a color, without going down. The gripper should be right over the chip.

In [ ]:
COLOR = 'red'
chips = look_for_chips(COLOR)
if chips:
    x, y = chips[0]['xy']
    move_to_xyz((x, y, HOVER_Z), GRIP_OPEN)

## 6. Gripper test
Finds `GRIP_OPEN` / `GRIP_CLOSED`. Hold a chip between the fingers and try values. Put the good ones in the settings cell.

In [ ]:
for g in [GRIP_OPEN, GRIP_CLOSED, GRIP_OPEN]:
    print("gripper ->", g)
    arm.Arm_serial_servo_write(6, g, 800)
    time.sleep(1.5)

## 7. Pick a chip!

In [ ]:
COLOR = 'red'   # 'red', 'green' or 'black'
picked = pick_chip(COLOR)

## 8. Drop it on the bet square
Set `BET_XY` in the settings cell first.

In [ ]:
if picked:
    place_on_bet()

## Done

In [ ]:
move(poses['home'], GRIP_OPEN)
cap.release()